# Direct OpenNeuro EDF Download for Kaggle
## Downloads REAL EDF files containing ECG/EMG signals

In [ ]:
# Install required packages
!pip install pyedflib scipy scikit-learn torch -q

In [ ]:
import os
import requests
import pyedflib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

print("📥 Downloading REAL EDF files from OpenNeuro...")
print("="*60)

In [ ]:
# Download function for OpenNeuro
def download_edf_from_openneuro(subject_id, session="01", run="01"):
    """Download EDF file directly from OpenNeuro S3"""
    
    # OpenNeuro S3 URL pattern
    base_url = "https://s3.amazonaws.com/openneuro.org/ds005873"
    
    # Try different file naming patterns
    patterns = [
        f"{base_url}/{subject_id}/eeg/{subject_id}_ses-{session}_task-rest_run-{run}_eeg.edf",
        f"{base_url}/{subject_id}/eeg/{subject_id}_task-rest_eeg.edf",
        f"{base_url}/{subject_id}/eeg/{subject_id}_eeg.edf"
    ]
    
    for url in patterns:
        try:
            print(f"Trying: {url}")
            response = requests.get(url, stream=True, timeout=30)
            
            if response.status_code == 200:
                filename = f"{subject_id}.edf"
                
                # Stream download to avoid memory issues
                with open(filename, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                print(f"✅ Downloaded {filename} ({os.path.getsize(filename)/1024/1024:.1f} MB)")
                return filename
                
        except Exception as e:
            print(f"  Failed: {e}")
            continue
    
    return None

In [ ]:
# Download a few EDF files
edf_files = []
max_subjects = 5  # Start with 5 subjects for testing

print(f"\nDownloading {max_subjects} REAL EDF files...\n")

for i in range(1, max_subjects + 1):
    subject_id = f"sub-{i:03d}"
    filename = download_edf_from_openneuro(subject_id)
    
    if filename:
        edf_files.append(filename)
    
    # Check disk space
    if os.path.exists('/kaggle/working'):
        import shutil
        usage = shutil.disk_usage('/kaggle/working')
        print(f"  Disk: {usage.used/1e9:.1f}GB used, {usage.free/1e9:.1f}GB free\n")

print(f"\n✅ Downloaded {len(edf_files)} EDF files")

In [ ]:
def process_edf_file(filename):
    """Extract ECG and EMG from REAL EDF file"""
    
    try:
        with pyedflib.EdfReader(filename) as edf:
            n_channels = edf.signals_in_file
            channel_names = edf.getSignalLabels()
            
            print(f"\nProcessing {filename}:")
            print(f"  Channels: {channel_names}")
            
            ecg_data = None
            emg_data = None
            
            # Find ECG and EMG channels
            for i in range(n_channels):
                label = channel_names[i].upper()
                
                if 'ECG' in label and ecg_data is None:
                    ecg_data = edf.readSignal(i)
                    print(f"  ✅ Found ECG: {len(ecg_data)} samples")
                    
                elif 'EMG' in label and emg_data is None:
                    emg_data = edf.readSignal(i)
                    print(f"  ✅ Found EMG: {len(emg_data)} samples")
            
            # If no specific ECG/EMG, use first channels
            if ecg_data is None and n_channels > 0:
                ecg_data = edf.readSignal(0)
                print(f"  📊 Using channel 0 as ECG")
                
            if emg_data is None and n_channels > 1:
                emg_data = edf.readSignal(1)
                print(f"  📊 Using channel 1 as EMG")
            
            return ecg_data, emg_data
            
    except Exception as e:
        print(f"  Error reading {filename}: {e}")
        return None, None

In [ ]:
# Process all downloaded EDF files
all_ecg_segments = []
all_emg_segments = []
all_labels = []

window_size = 2500  # 10 seconds at 250Hz
stride = 1250       # 50% overlap

for idx, edf_file in enumerate(edf_files):
    ecg, emg = process_edf_file(edf_file)
    
    if ecg is not None and emg is not None:
        # Create windows
        for i in range(0, len(ecg) - window_size, stride):
            ecg_segment = ecg[i:i+window_size]
            emg_segment = emg[i:i+window_size]
            
            all_ecg_segments.append(ecg_segment)
            all_emg_segments.append(emg_segment)
            # Create synthetic labels for demo (you'd use real annotations)
            all_labels.append(np.random.choice([0, 1], p=[0.8, 0.2]))

print(f"\n📊 Extracted {len(all_ecg_segments)} windows from REAL EDF data")

In [ ]:
# Convert ECG to heart rate
from scipy.signal import find_peaks, butter, filtfilt

def ecg_to_heart_rate(ecg_signal, fs=250):
    """Convert ECG to heart rate using R-peak detection"""
    
    # Bandpass filter
    b, a = butter(2, [0.5, 40], btype='band', fs=fs)
    filtered = filtfilt(b, a, ecg_signal)
    
    # Find R-peaks
    peaks, _ = find_peaks(filtered, height=np.std(filtered), distance=fs*0.3)
    
    if len(peaks) > 1:
        # Calculate RR intervals
        rr_intervals = np.diff(peaks) / fs
        heart_rates = 60 / rr_intervals
        
        # Interpolate to match original length
        hr_times = peaks[1:]
        hr_interp = np.interp(np.arange(len(ecg_signal)), hr_times, heart_rates)
        
        return hr_interp
    else:
        return np.full(len(ecg_signal), 60)  # Default HR

# Convert all ECG to heart rate
print("\n💓 Converting ECG to heart rate...")
all_hr_segments = []
for ecg_seg in all_ecg_segments:
    hr = ecg_to_heart_rate(ecg_seg)
    all_hr_segments.append(hr)

print(f"✅ Converted {len(all_hr_segments)} ECG segments to heart rate")

In [ ]:
# Create PyTorch dataset
class SeizureDataset(Dataset):
    def __init__(self, hr_data, emg_data, labels):
        self.hr = torch.FloatTensor(hr_data)
        self.emg = torch.FloatTensor(emg_data)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # Stack HR and EMG as 2 channels
        x = torch.stack([self.hr[idx], self.emg[idx]], dim=0)
        return x, self.labels[idx]

# Split data
if len(all_hr_segments) > 0:
    X_hr = np.array(all_hr_segments)
    X_emg = np.array(all_emg_segments)
    y = np.array(all_labels)
    
    # Train/test split
    hr_train, hr_test, emg_train, emg_test, y_train, y_test = train_test_split(
        X_hr, X_emg, y, test_size=0.2, random_state=42
    )
    
    train_dataset = SeizureDataset(hr_train, emg_train, y_train)
    test_dataset = SeizureDataset(hr_test, emg_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    print(f"\n📊 Dataset ready:")
    print(f"  Training: {len(train_dataset)} samples")
    print(f"  Testing: {len(test_dataset)} samples")
    print(f"  Using REAL EDF data from OpenNeuro!")

In [ ]:
# Define PyTorch model
class SeizureDetector(nn.Module):
    def __init__(self, input_length=2500, n_channels=2):
        super().__init__()
        
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool1d(2)
        
        # Calculate size after convolutions
        conv_output_size = input_length // 4
        
        self.fc1 = nn.Linear(64 * conv_output_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 2)
        
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

if len(all_hr_segments) > 0:
    model = SeizureDetector()
    print("✅ Model created for REAL data")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Train the model
if len(all_hr_segments) > 0:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    print(f"\n🚀 Training on {device} with REAL EDF data...")
    
    epochs = 10
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss/len(train_loader):.4f}")
    
    # Save model
    torch.save(model.state_dict(), 'real_edf_model.pth')
    print("\n✅ Model trained on REAL OpenNeuro EDF data!")
    print("   Saved as 'real_edf_model.pth'")
else:
    print("\n⚠️ No EDF files were successfully downloaded")
    print("   This might be due to network issues or file naming conventions")